# MR1b — Hourly Crash MapReduce
**Input**: `crash_manhattan_alcohol_clean.csv`
**Output**: `hourly_crash_mr.csv`

Schema: `zone_id, hour, avg_crashes, avg_injured, avg_killed, avg_lat, avg_lon`

In [1]:
import h3
import pandas as pd
import numpy as np
from collections import defaultdict

INPUT  = 'crash_manhattan_alcohol_clean.csv'
OUTPUT = 'hourly_crash_mr.csv'
H3_RES = 10
LAT_BOUNDS = (40.70, 40.88)
LON_BOUNDS = (-74.02, -73.91)

df = pd.read_csv(INPUT, dtype=str, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['number_of_persons_injured'] = pd.to_numeric(df.get('number_of_persons_injured'), errors='coerce').fillna(0)
df['number_of_persons_killed']  = pd.to_numeric(df.get('number_of_persons_killed'),  errors='coerce').fillna(0)

df = df[
    df['latitude'].between(*LAT_BOUNDS) &
    df['longitude'].between(*LON_BOUNDS)
].copy()

df['zone_id'] = df.apply(
    lambda r: h3.latlng_to_cell(r['latitude'], r['longitude'], H3_RES), axis=1
)

def parse_hour(t):
    try: return int(str(t).split(':')[0])
    except: return 0

df['hour'] = df['crash_time'].apply(parse_hour)

print(f'Input rows (clean, in-bounds): {len(df):,}')
print(f'Unique zones: {df["zone_id"].nunique():,}')

Input rows (clean, in-bounds): 2,887
Unique zones: 1,488


In [2]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# key=(zone_id, hour), val=1 crash + injured + killed

def mapper(row):
    key = (str(row['zone_id']), int(row['hour']))
    val = {
        'crashes': 1,
        'injured': float(row['number_of_persons_injured']),
        'killed':  float(row['number_of_persons_killed']),
        'lat':     row['latitude'],
        'lon':     row['longitude'],
        'date':    str(row.get('crash_date', '')),
    }
    return key, val

mapped = [mapper(row) for _, row in df.iterrows()]
print(f'Mapped pairs: {len(mapped):,}')

Mapped pairs: 2,887


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Per (zone_id, hour): sum totals + track unique dates → compute true avg

acc = defaultdict(lambda: {'crashes':0,'injured':0.0,'killed':0.0,'lat':[],'lon':[],'dates':set()})

for key, val in mapped:
    a = acc[key]
    a['crashes'] += val['crashes']
    a['injured'] += val['injured']
    a['killed']  += val['killed']
    a['dates'].add(val['date'])
    if pd.notna(val['lat']): a['lat'].append(float(val['lat']))
    if pd.notna(val['lon']): a['lon'].append(float(val['lon']))

rows = []
for (zone_id, hour), a in acc.items():
    n = max(len(a['dates']), 1)
    rows.append({
        'zone_id':     zone_id,
        'hour':        hour,
        'avg_crashes': round(a['crashes'] / n, 4),
        'avg_injured': round(a['injured'] / n, 4),
        'avg_killed':  round(a['killed']  / n, 4),
        'avg_lat':     float(np.mean(a['lat'])) if a['lat'] else np.nan,
        'avg_lon':     float(np.mean(a['lon'])) if a['lon'] else np.nan,
    })

# ── OUTPUT ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(rows).sort_values(['zone_id','hour']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Hours covered: {sorted(out["hour"].unique())}')
print(f'Saved → {OUTPUT}')
out.head(10)

Output rows : 2,696
Unique zones: 1,488
Hours covered: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Saved → hourly_crash_mr.csv


,zone_id,hour,avg_crashes,avg_injured,avg_killed,avg_lat,avg_lon
0,8a2a1008800ffff,1,1.0,2.0,0.0,40.796528,-73.974120
1,8a2a1008800ffff,20,1.0,1.0,0.0,40.796307,-73.974434
2,8a2a1008802ffff,0,1.0,0.0,0.0,40.797886,-73.973465
3,8a2a1008802ffff,23,1.0,1.0,0.0,40.797150,-73.973660
4,8a2a10088047fff,3,1.0,0.0,0.0,40.795250,-73.973210
5,8a2a10088047fff,17,1.0,0.0,0.0,40.795252,-73.973209
6,8a2a10088047fff,20,1.0,0.0,0.0,40.794994,-73.972590
7,8a2a1008804ffff,6,1.0,0.0,0.0,40.794651,-73.971783
8,8a2a1008804ffff,21,1.0,1.0,0.0,40.794650,-73.971794
9,8a2a100880e7fff,14,1.0,0.0,0.0,40.795532,-73.975845
